# Advanced Problems with Solutions: Coercing Floats to Integers

Python version target: **Python 3.13**

Topics covered:
- `int()` and `math.trunc()`
- `math.floor()`
- `math.ceil()`
- Negative numbers
- Floating-point surprises
- Custom numeric classes
- Safe integer coercion patterns
- Real-world indexing and pagination problems

## Setup

In [1]:
from math import trunc, floor, ceil, isclose
from decimal import Decimal, ROUND_FLOOR, ROUND_CEILING, ROUND_DOWN
from fractions import Fraction
import math

## Problem 1: Predict the Results

Without running the code first, predict the output.

```python
values = [3.9, 3.1, -3.1, -3.9, 0.0, -0.0]
[(x, int(x), trunc(x), floor(x), ceil(x)) for x in values]
```

In [2]:
values = [3.9, 3.1, -3.1, -3.9, 0.0, -0.0]
result = [(x, int(x), trunc(x), floor(x), ceil(x)) for x in values]
result

[(3.9, 3, 3, 3, 4),
 (3.1, 3, 3, 3, 4),
 (-3.1, -3, -3, -4, -3),
 (-3.9, -3, -3, -4, -3),
 (0.0, 0, 0, 0, 0),
 (-0.0, 0, 0, 0, 0)]

### Solution

`int(x)` and `trunc(x)` both remove the fractional part and move toward zero.

`floor(x)` moves toward negative infinity.

`ceil(x)` moves toward positive infinity.

So for negative numbers, `floor()` and `int()` often differ.

## Problem 2: Implement `ceil_division`

Write a function `ceil_division(a, b)` that returns the smallest integer greater than or equal to `a / b`.

Requirements:
- Work for positive and negative integers.
- Do not use floating-point division.
- Raise `ZeroDivisionError` naturally when `b == 0`.

In [3]:
def ceil_division(a: int, b: int) -> int:
    return -(-a // b)


tests = [
    (10, 3),
    (9, 3),
    (-10, 3),
    (10, -3),
    (-10, -3),
]

[(a, b, ceil_division(a, b), ceil(a / b)) for a, b in tests]

[(10, 3, 4, 4),
 (9, 3, 3, 3),
 (-10, 3, -3, -3),
 (10, -3, -3, -3),
 (-10, -3, 4, 4)]

### Solution

Python's `//` operator performs floor division, not truncating division.

The identity

```python
ceil(a / b) == -((-a) // b)
```

works for both positive and negative values of `a` and `b`.

## Problem 3: Bug Hunt — Pagination

A developer writes this function:

```python
def number_of_pages(items, page_size):
    return int(items / page_size)
```

Find the bug and fix it.

Examples:
- `number_of_pages(100, 10)` should be `10`
- `number_of_pages(101, 10)` should be `11`
- `number_of_pages(0, 10)` should be `0`

In [4]:
def number_of_pages(items: int, page_size: int) -> int:
    if items < 0:
        raise ValueError("items must be non-negative")
    if page_size <= 0:
        raise ValueError("page_size must be positive")
    return ceil_division(items, page_size)


assert number_of_pages(100, 10) == 10
assert number_of_pages(101, 10) == 11
assert number_of_pages(0, 10) == 0
assert number_of_pages(1, 10) == 1

number_of_pages(101, 10)

11

### Solution

`int(items / page_size)` truncates toward zero, so it loses the extra partially filled page.

For pagination, we need ceiling division.

## Problem 4: Negative Bucket Indexing

Suppose numbers are assigned to buckets of width `10`.

Write a function `bucket_index(x)` such that:

- `0 <= x < 10` maps to `0`
- `10 <= x < 20` maps to `1`
- `-10 <= x < 0` maps to `-1`
- `-20 <= x < -10` maps to `-2`

Which coercion method should you use?

In [5]:
def bucket_index(x: float, width: float = 10.0) -> int:
    if width <= 0:
        raise ValueError("width must be positive")
    return floor(x / width)


examples = [-20.1, -20, -19.9, -10.1, -10, -9.9, -0.1, 0, 0.1, 9.9, 10, 10.1]
[(x, bucket_index(x)) for x in examples]

[(-20.1, -3),
 (-20, -2),
 (-19.9, -2),
 (-10.1, -2),
 (-10, -1),
 (-9.9, -1),
 (-0.1, -1),
 (0, 0),
 (0.1, 0),
 (9.9, 0),
 (10, 1),
 (10.1, 1)]

### Solution

Use `floor()`, not `int()`.

`int(-0.1)` is `0`, but for bucket indexing, `-0.1` belongs in bucket `-1`.

Flooring is the correct operation because bucket intervals are usually half-open intervals extending to the right.

## Problem 5: Floating-Point Surprise

Explain why this may be dangerous:

```python
int(0.1 + 0.2 + 0.3)
```

Then write a safer version when the intended result is based on exact decimal arithmetic.

In [6]:
raw = 0.1 + 0.2 + 0.3
raw, int(raw)

(0.6000000000000001, 0)

In [7]:
exact = Decimal("0.1") + Decimal("0.2") + Decimal("0.3")
exact, int(exact)

(Decimal('0.6'), 0)

### Solution

Binary floating-point numbers cannot represent many decimal fractions exactly.

For exact decimal calculations, use `Decimal` with string inputs.

Do not write:

```python
Decimal(0.1)
```

because that first creates the already-imprecise float.

## Problem 6: Custom Numeric Type

Create a class `TemperatureReading` that wraps a float and supports:

- `int(obj)` using truncation
- `math.floor(obj)`
- `math.ceil(obj)`
- `math.trunc(obj)`

Then test it with positive and negative readings.

In [8]:
class TemperatureReading:
    def __init__(self, value: float):
        self.value = float(value)

    def __int__(self) -> int:
        return trunc(self.value)

    def __trunc__(self) -> int:
        return trunc(self.value)

    def __floor__(self) -> int:
        return floor(self.value)

    def __ceil__(self) -> int:
        return ceil(self.value)

    def __repr__(self) -> str:
        return f"TemperatureReading({self.value})"


readings = [TemperatureReading(21.9), TemperatureReading(-21.9)]

[(r, int(r), trunc(r), floor(r), ceil(r)) for r in readings]

[(TemperatureReading(21.9), 21, 21, 21, 22),
 (TemperatureReading(-21.9), -21, -21, -22, -21)]

### Solution

`int(obj)` calls `obj.__int__()` when available.

`math.trunc(obj)` calls `obj.__trunc__()`.

`math.floor(obj)` calls `obj.__floor__()`.

`math.ceil(obj)` calls `obj.__ceil__()`.

## Problem 7: Validate Safe Integer Conversion

Write a function `to_exact_int(x)` that converts `x` to an integer only if it is mathematically integral.

Examples:

- `10.0` becomes `10`
- `10.5` raises `ValueError`
- `Decimal("10.00")` becomes `10`
- `Fraction(20, 2)` becomes `10`
- `Fraction(21, 2)` raises `ValueError`

In [9]:
def to_exact_int(x) -> int:
    if isinstance(x, bool):
        raise TypeError("bool is not accepted")

    if isinstance(x, int):
        return x

    if isinstance(x, float):
        if not math.isfinite(x):
            raise ValueError("cannot convert infinity or NaN to int")
        if x.is_integer():
            return int(x)
        raise ValueError(f"not an exact integer: {x!r}")

    if isinstance(x, Decimal):
        if x.is_finite() and x == x.to_integral_value():
            return int(x)
        raise ValueError(f"not an exact integer: {x!r}")

    if isinstance(x, Fraction):
        if x.denominator == 1:
            return x.numerator
        raise ValueError(f"not an exact integer: {x!r}")

    raise TypeError(f"unsupported type: {type(x).__name__}")


valid_values = [10, 10.0, Decimal("10.00"), Fraction(20, 2)]
[to_exact_int(x) for x in valid_values]

[10, 10, 10, 10]

In [10]:
invalid_values = [10.5, Decimal("10.01"), Fraction(21, 2), float("inf"), float("nan")]

for value in invalid_values:
    try:
        print(value, "->", to_exact_int(value))
    except Exception as exc:
        print(value, "->", type(exc).__name__, exc)

10.5 -> ValueError not an exact integer: 10.5
10.01 -> ValueError not an exact integer: Decimal('10.01')
21/2 -> ValueError not an exact integer: Fraction(21, 2)
inf -> ValueError cannot convert infinity or NaN to int
nan -> ValueError cannot convert infinity or NaN to int


### Solution

This is safer than blindly calling `int(x)`, because `int(10.9)` silently returns `10`.

In production code, silent truncation can hide data loss.

## Problem 8: Rounding Is Not Coercion

Compare the behavior of `round`, `int`, `floor`, and `ceil` for these values:

```python
[2.5, 3.5, -2.5, -3.5]
```

Explain why `round()` should not be used as a replacement for integer coercion.

In [11]:
values = [2.5, 3.5, -2.5, -3.5]
[(x, round(x), int(x), floor(x), ceil(x)) for x in values]

[(2.5, 2, 2, 2, 3),
 (3.5, 4, 3, 3, 4),
 (-2.5, -2, -2, -3, -2),
 (-3.5, -4, -3, -4, -3)]

### Solution

`round()` uses bankers' rounding for ties: halves are rounded to the nearest even integer.

So `round(2.5)` is `2`, while `round(3.5)` is `4`.

This is different from truncation, floor, and ceiling.

## Problem 9: Decimal Rounding Modes

Using `Decimal`, show the difference between:

- rounding down toward zero
- rounding toward negative infinity
- rounding toward positive infinity

Use the values `Decimal("10.9")` and `Decimal("-10.9")`.

In [12]:
values = [Decimal("10.9"), Decimal("-10.9")]

rows = []
for x in values:
    rows.append((
        x,
        x.to_integral_value(rounding=ROUND_DOWN),
        x.to_integral_value(rounding=ROUND_FLOOR),
        x.to_integral_value(rounding=ROUND_CEILING),
    ))

rows

[(Decimal('10.9'), Decimal('10'), Decimal('10'), Decimal('11')),
 (Decimal('-10.9'), Decimal('-10'), Decimal('-11'), Decimal('-10'))]

### Solution

`ROUND_DOWN` means toward zero.

`ROUND_FLOOR` means toward negative infinity.

`ROUND_CEILING` means toward positive infinity.

These names are especially important for negative numbers.

## Problem 10: Safe Pixel Coordinates

You are building a graphics system. A mouse click gives a floating-point coordinate.

Write a function `pixel_at(x, y)` that maps coordinates to pixel cells.

Assume pixel cells are:

- pixel `0` covers `[0, 1)`
- pixel `1` covers `[1, 2)`
- pixel `-1` covers `[-1, 0)`

Return a tuple `(px, py)`.

In [13]:
def pixel_at(x: float, y: float) -> tuple[int, int]:
    return floor(x), floor(y)


points = [
    (0.0, 0.0),
    (0.999, 0.999),
    (1.0, 1.0),
    (-0.001, -0.001),
    (-1.0, -1.0),
]

[(p, pixel_at(*p)) for p in points]

[((0.0, 0.0), (0, 0)),
 ((0.999, 0.999), (0, 0)),
 ((1.0, 1.0), (1, 1)),
 ((-0.001, -0.001), (-1, -1)),
 ((-1.0, -1.0), (-1, -1))]

### Solution

Use `floor()`, because pixel cells are half-open intervals of the form `[n, n + 1)`.

`int(-0.001)` would incorrectly return `0`.

## Problem 11: Coercion Table Generator

Write a function that generates a comparison table for a list of floats.

The table should include:

- original value
- `int(x)`
- `trunc(x)`
- `floor(x)`
- `ceil(x)`
- whether `int(x) == floor(x)`

In [14]:
def coercion_table(values):
    return [
        {
            "x": x,
            "int": int(x),
            "trunc": trunc(x),
            "floor": floor(x),
            "ceil": ceil(x),
            "int_equals_floor": int(x) == floor(x),
        }
        for x in values
    ]


coercion_table([-2.7, -2.0, -0.1, 0.0, 0.1, 2.7])

[{'x': -2.7,
  'int': -2,
  'trunc': -2,
  'floor': -3,
  'ceil': -2,
  'int_equals_floor': False},
 {'x': -2.0,
  'int': -2,
  'trunc': -2,
  'floor': -2,
  'ceil': -2,
  'int_equals_floor': True},
 {'x': -0.1,
  'int': 0,
  'trunc': 0,
  'floor': -1,
  'ceil': 0,
  'int_equals_floor': False},
 {'x': 0.0,
  'int': 0,
  'trunc': 0,
  'floor': 0,
  'ceil': 0,
  'int_equals_floor': True},
 {'x': 0.1,
  'int': 0,
  'trunc': 0,
  'floor': 0,
  'ceil': 1,
  'int_equals_floor': True},
 {'x': 2.7,
  'int': 2,
  'trunc': 2,
  'floor': 2,
  'ceil': 3,
  'int_equals_floor': True}]

### Solution

`int(x) == floor(x)` is true for positive floats and already-integral negative floats.

It is false for negative floats with a fractional part.

## Problem 12: Detect Truncation Loss

Write a function `truncation_loss(x)` that returns the part removed by truncation.

Examples:

- `truncation_loss(10.75)` should be `0.75`
- `truncation_loss(-10.75)` should be `-0.75`

Then explain why the sign matters.

In [15]:
def truncation_loss(x: float) -> float:
    return x - trunc(x)


[(x, trunc(x), truncation_loss(x)) for x in [10.75, -10.75, 0.25, -0.25]]

[(10.75, 10, 0.75), (-10.75, -10, -0.75), (0.25, 0, 0.25), (-0.25, 0, -0.25)]

### Solution

The removed part has the same sign as the original number.

For negative numbers, truncation moves the value upward toward zero.

## Problem 13: Large Float Conversion

Investigate this expression:

```python
int(10**20 + 0.9)
```

Why can this be surprising?

In [16]:
x = 10**20 + 0.9
x, int(x), x == 10**20

(1e+20, 100000000000000000000, True)

### Solution

`10**20 + 0.9` is evaluated as a float because of `0.9`.

At that magnitude, a float cannot represent every integer, much less small fractional changes.

The `0.9` is lost before `int()` is ever called.

## Problem 14: Choose the Correct Coercion

For each scenario, choose `int`, `trunc`, `floor`, or `ceil`.

1. Removing the fractional part of a sensor reading.
2. Finding the grid cell containing coordinate `x`.
3. Computing the number of pages needed for `n` items.
4. Finding the next integer timestamp not earlier than a given time.
5. Implementing mathematical floor.

### Solution

1. `trunc` or `int`, if silent loss is acceptable.
2. `floor`, especially if negative coordinates exist.
3. `ceil`, preferably via integer arithmetic.
4. `ceil`.
5. `floor`.

Best practice: choose the function whose mathematical meaning matches the problem. Do not choose based only on examples with positive numbers.

## Problem 15: Mini Project — Robust Bin Counter

Write a function `count_bins(values, width)` that counts how many values fall into each bin.

Bins should be indexed using floor-based binning:

```python
bin_index = floor(value / width)
```

Example:

```python
values = [-10.1, -10.0, -9.9, -0.1, 0.0, 0.1, 9.9, 10.0]
width = 10
```

Expected bins:

```python
{-2: 1, -1: 3, 0: 3, 1: 1}
```

In [17]:
def count_bins(values, width: float) -> dict[int, int]:
    if width <= 0:
        raise ValueError("width must be positive")

    counts: dict[int, int] = {}

    for value in values:
        index = floor(value / width)
        counts[index] = counts.get(index, 0) + 1

    return counts


values = [-10.1, -10.0, -9.9, -0.1, 0.0, 0.1, 9.9, 10.0]
count_bins(values, 10)

{-2: 1, -1: 3, 0: 3, 1: 1}

### Solution

`floor(value / width)` correctly handles both positive and negative values.

Using `int(value / width)` would incorrectly place values such as `-0.1` into bin `0` instead of bin `-1`.

## Final Best Practices Summary

- Use `int(x)` when you explicitly want truncation toward zero.
- Use `math.trunc(x)` when you want to communicate truncation clearly.
- Use `math.floor(x)` for grid cells, bins, and lower bounds.
- Use `math.ceil(x)` for required capacity, pages, chunks, or upper bounds.
- Avoid float arithmetic when exact integer results are required.
- Prefer integer formulas such as `-(-a // b)` for ceiling division.
- Validate before converting if fractional data loss would be a bug.
- Be especially careful with negative numbers.